# PySYD Power Spectra — All 26 Targets
Shows the background-corrected power spectrum for each target with numax highlighted.
Run all cells to generate all plots.


> **Revision update (2026):** PySYD provides the initial $\nu_{\max}$; the final per-target diagnostic panels (Appendix C) refit the background with a two-component Harvey profile (Kallinger et al. 2014) in log space over the full spectrum with the oscillation region masked — see `scripts/diagnostic_panels.py`. TESS-SPOC data are preferred over QLP where available.

In [ ]:
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
os.chdir('/Users/carlimankowski/research')

import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, gaussian_filter1d

def plot_pysyd_ps(tic, numax, dnu_target, cluster):
    """Plot power spectrum with numax region highlighted."""
    data = np.loadtxt(f"data/{tic}_PS.txt")
    freq, power = data[:,0], data[:,1]

    # Useful range
    fmin = max(0.1, numax * 0.05)
    fmax = min(numax * 6, 300)
    mask = (freq >= fmin) & (freq <= fmax)
    freq, power = freq[mask], power[mask]

    # Background via median filter
    df = np.median(np.diff(freq))
    kern = max(11, int(numax * 0.5 / df))
    if kern % 2 == 0: kern += 1
    kern = min(kern, max(3, len(power)//2))
    bg = median_filter(power, size=kern)
    snr = power / np.maximum(bg, 1e-10)

    # Smooth for display
    smooth_kern = max(3, int(dnu_target * 0.3 / df))
    snr_smooth = gaussian_filter1d(snr, smooth_kern)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: full power spectrum + background
    ax = axes[0]
    # Bin for display
    nbins = min(2000, len(freq)//5)
    if nbins > 10:
        bin_edges = np.linspace(freq[0], freq[-1], nbins+1)
        f_b, p_b, bg_b = [], [], []
        for i in range(nbins):
            m = (freq >= bin_edges[i]) & (freq < bin_edges[i+1])
            if m.sum() > 0:
                f_b.append(freq[m].mean())
                p_b.append(power[m].mean())
                bg_b.append(bg[m].mean())
        ax.semilogy(f_b, p_b, 'k-', lw=0.5, alpha=0.6)
        ax.semilogy(f_b, bg_b, 'r-', lw=2, label='Background')
    else:
        ax.semilogy(freq, power, 'k-', lw=0.5, alpha=0.6)
        ax.semilogy(freq, bg, 'r-', lw=2, label='Background')

    # Highlight numax region
    ax.axvspan(numax - 2*dnu_target, numax + 2*dnu_target,
               alpha=0.2, color='green', label=f'numax region')
    ax.axvline(numax, color='blue', ls='--', lw=2,
               label=f'numax = {numax:.1f} uHz')
    ax.set_xlabel('Frequency [uHz]')
    ax.set_ylabel('Power')
    ax.set_title(f'TIC {tic} ({cluster}) — Power Spectrum')
    ax.legend(fontsize=9)

    # Right: SNR spectrum zoomed on numax
    ax = axes[1]
    win = 5 * dnu_target
    wm = (freq >= numax - win) & (freq <= numax + win)
    if wm.sum() > 0:
        ax.plot(freq[wm], snr[wm], 'k-', lw=0.4, alpha=0.4)
        ax.plot(freq[wm], snr_smooth[wm], 'b-', lw=2, label='Smoothed SNR')
        ax.axvline(numax, color='red', ls='--', lw=2,
                   label=f'numax = {numax:.1f}')
        ax.axhline(1, color='gray', ls=':', lw=1)

        # Mark the peak
        peak_idx = np.argmax(snr_smooth[wm])
        peak_freq = freq[wm][peak_idx]
        peak_val = snr_smooth[wm][peak_idx]
        ax.plot(peak_freq, peak_val, 'rv', ms=12,
                label=f'Peak = {peak_freq:.1f} uHz')

    ax.set_xlabel('Frequency [uHz]')
    ax.set_ylabel('SNR')
    ax.set_title(f'Background-Corrected — numax region')
    ax.legend(fontsize=9)

    plt.suptitle(f'TIC {tic} — {cluster} | PySYD numax = {numax:.1f} uHz | '
                 f'target dnu = {dnu_target:.3f} uHz',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

print(f"Setup OK. {26} targets ready.")


### TIC 352774198 — COIN-Gaia_30 | numax = 3.1 uHz


In [ ]:
plot_pysyd_ps(352774198, 3.08, 0.632, 'COIN-Gaia_30')


### TIC 306955660 — Casado_Alessi_1 | numax = 62.9 uHz


In [ ]:
plot_pysyd_ps(306955660, 62.90, 5.381, 'Casado_Alessi_1')


### TIC 306552813 — Casado_Alessi_1 | numax = 66.2 uHz


In [ ]:
plot_pysyd_ps(306552813, 66.16, 5.627, 'Casado_Alessi_1')


### TIC 306187638 — Casado_Alessi_1 | numax = 68.1 uHz


In [ ]:
plot_pysyd_ps(306187638, 68.06, 5.737, 'Casado_Alessi_1')


### TIC 140143506 — HSC_1986 | numax = 171.8 uHz


In [ ]:
plot_pysyd_ps(140143506, 171.76, 11.403, 'HSC_1986')


### TIC 402808611 — HSC_2615 | numax = 4.2 uHz


In [ ]:
plot_pysyd_ps(402808611, 4.21, 0.615, 'HSC_2615')


### TIC 414752508 — HSC_95 | numax = 28.0 uHz


In [ ]:
plot_pysyd_ps(414752508, 27.97, 3.411, 'HSC_95')


### TIC 354016987 — LISC_3534 | numax = 4.5 uHz


In [ ]:
plot_pysyd_ps(354016987, 4.51, 0.826, 'LISC_3534')


### TIC 437030530 — NGC_2682 | numax = 4.2 uHz


In [ ]:
plot_pysyd_ps(437030530, 4.23, 0.781, 'NGC_2682')


### TIC 67419922 — NGC_752 | numax = 33.1 uHz


In [ ]:
plot_pysyd_ps(67419922, 33.11, 3.286, 'NGC_752')


### TIC 186970424 — NGC_752 | numax = 59.6 uHz


In [ ]:
plot_pysyd_ps(186970424, 59.58, 5.290, 'NGC_752')


### TIC 67569102 — NGC_752 | numax = 64.7 uHz


In [ ]:
plot_pysyd_ps(67569102, 64.70, 6.820, 'NGC_752')


### TIC 67420118 — NGC_752 | numax = 75.0 uHz


In [ ]:
plot_pysyd_ps(67420118, 75.02, 6.610, 'NGC_752')


### TIC 150753375 — Theia_1188 | numax = 5.5 uHz


In [ ]:
plot_pysyd_ps(150753375, 5.50, 0.960, 'Theia_1188')


### TIC 321823630 — Theia_1297 | numax = 26.8 uHz


In [ ]:
plot_pysyd_ps(321823630, 26.83, 3.284, 'Theia_1297')


### TIC 43902016 — Theia_6046 | numax = 3.3 uHz


In [ ]:
plot_pysyd_ps(43902016, 3.28, 0.558, 'Theia_6046')


### TIC 24444542 — Theia_6046 | numax = 5.0 uHz


In [ ]:
plot_pysyd_ps(24444542, 4.98, 0.749, 'Theia_6046')


### TIC 24666306 — Theia_6046 | numax = 16.9 uHz


In [ ]:
plot_pysyd_ps(24666306, 16.86, 1.871, 'Theia_6046')


### TIC 24297458 — Theia_6046 | numax = 32.1 uHz


In [ ]:
plot_pysyd_ps(24297458, 32.10, 3.083, 'Theia_6046')


### TIC 34472483 — Theia_6046 | numax = 33.5 uHz


In [ ]:
plot_pysyd_ps(34472483, 33.46, 3.293, 'Theia_6046')


### TIC 306345133 — Theia_6046 | numax = 34.6 uHz


In [ ]:
plot_pysyd_ps(306345133, 34.60, 3.251, 'Theia_6046')


### TIC 249064439 — Theia_6046 | numax = 37.7 uHz


In [ ]:
plot_pysyd_ps(249064439, 37.70, 3.638, 'Theia_6046')


### TIC 195862014 — Theia_844 | numax = 28.7 uHz


In [ ]:
plot_pysyd_ps(195862014, 28.73, 2.873, 'Theia_844')


### TIC 195860623 — Theia_844 | numax = 37.2 uHz


In [ ]:
plot_pysyd_ps(195860623, 37.21, 3.504, 'Theia_844')


### TIC 427725221 — UPK_226 | numax = 2.8 uHz


In [ ]:
plot_pysyd_ps(427725221, 2.80, 0.584, 'UPK_226')


### TIC 459055617 — Unknown_3 | numax = 5.1 uHz


In [ ]:
plot_pysyd_ps(459055617, 5.10, 0.907, 'Unknown_3')
